In [1]:
from pathlib import Path
import subprocess

import joblib
import mlflow
import pandas as pd
from lightgbm import LGBMClassifier

from qrt.data import (
    load_tr_data,
    load_te_data,
    load_sample_submission_data,
)
from qrt.features import compute_features

root = Path("..").resolve()
data_dir = root / "data/raw"

mlflow.set_tracking_uri("http://127.0.0.1:5001")
mlflow.set_experiment("qrt-aapf")

cv_run_id = "f450faaa6dd044eeaa20482551cf260c"
threshold = 0.5

In [2]:
X_tr, y_tr = load_tr_data(data_dir)
X_te = load_te_data(data_dir)
sample_submission = load_sample_submission_data(data_dir)

In [3]:
X = compute_features(X_tr)
X = X.drop(columns=["TS"])
X = X.drop(columns=X.filter(regex="^TS_").columns)

# Match the training columns and the submission row order
X_test = compute_features(X_te)
X_test = X_test.loc[sample_submission.index, X.columns].copy()

# Use the same category definitions in train and test
for col in ["GROUP", "ALLOCATION"]:
    X[col] = X[col].astype("category")
    X_test[col] = X_test[col].astype(X[col].dtype)

y = (y_tr.loc[X.index, "target"] > 0).astype(int)

assert X.shape[1] == 55
assert X.columns.equals(X_test.columns)
assert X.index.equals(y.index)

X.shape, X_test.shape

((527073, 55), (31870, 55))

In [4]:
model = LGBMClassifier(
    n_estimators=200,
    num_leaves=7,
    learning_rate=0.05,
    random_state=42,
    verbosity=-1,
    cat_smooth=1000,
)

In [5]:
with mlflow.start_run(run_name="lgbm_submission_v1") as run:
    mlflow.set_tags({
        "stage": "submission",
        "model": type(model).__name__,
        "source_cv_run_id": cv_run_id,
        "reason": "Fit the selected configuration on all labeled data",
        "git_commit": subprocess.check_output(
            ["git", "rev-parse", "HEAD"],
            cwd=root,
            text=True,
        ).strip(),
    })

    mlflow.log_params(model.get_params())
    mlflow.log_params({
        "n_features": X.shape[1],
        "n_train": len(X),
        "n_test": len(X_test),
        "prediction_threshold": threshold,
    })

    mlflow.log_dict({
        "features": X.columns.tolist(),
        "categorical_features": ["GROUP", "ALLOCATION"],
        "target": "target > 0",
        "training_files": ["X_train.csv", "y_train.csv"],
        "test_file": "X_test.csv",
    }, "config.json")

    # Save the source code and environment
    for path in (root / "src/qrt").glob("*.py"):
        mlflow.log_artifact(str(path), artifact_path="source/qrt")

    for name in (
        "pyproject.toml",
        "uv.lock",
        "notebooks/03_submissions.ipynb",
    ):
        mlflow.log_artifact(str(root / name), artifact_path="source")

    # Fit on all labeled observations
    model.fit(X, y)

    probabilities = pd.Series(
        model.predict_proba(X_test)[:, 1],
        index=X_test.index,
        name="probability",
    )
    assert probabilities.between(0, 1).all()

    # Preserve the sample submission format
    submission = sample_submission.copy()
    submission["prediction"] = (probabilities >= threshold).astype(int)

    assert submission.index.equals(sample_submission.index)
    assert submission.shape == sample_submission.shape
    assert submission["prediction"].isin([0, 1]).all()

    # Keep a separate directory for each submission run
    output_dir = root / "outputs/submissions" / run.info.run_id
    output_dir.mkdir(parents=True, exist_ok=True)

    submission_path = output_dir / "submission.csv"
    submission.to_csv(submission_path, index_label="ROW_ID")
    joblib.dump(model, output_dir / "model.joblib")

    mlflow.log_artifact(str(submission_path))
    mlflow.log_artifact(str(output_dir / "model.joblib"))
    mlflow.log_text(probabilities.to_csv(), "test_probabilities.csv")

print(f"Submission file: {submission_path}")
submission.head()

🏃 View run lgbm_submission_v1 at: http://127.0.0.1:5001/#/experiments/2/runs/5a0255575ffa4f9aa39eeef3cf8fc2bf
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/2
Submission file: /Users/benjaminemily/Developer/Project/qrt aapf/outputs/submissions/5a0255575ffa4f9aa39eeef3cf8fc2bf/submission.csv


,prediction
ROW_ID,
527073,1
527074,1
527075,0
527076,0
527077,0
